# LeapQuant 实验(二): triton-ascend 内核路径 (PR2 / M3.1-M3.4)

PR1 (torch) 的数值是 golden; PR2 把同一套数学下沉为 **triton-ascend 融合内核** (分支 `leapquant-triton-m3`):

| 里程碑 | 内核 | 本 notebook 演示 |
|---|---|---|
| M3.1 | `_leapquant_flush_scales/reconstruct_kernel` | 融合 flush vs torch golden (round-trip 模式) |
| M3.2 | `_leapquant_payload_encode_kernel` + payload 池 | 1.19 B/elem checkpoint, 双写语义, 内存压缩比 |
| M3.3 | `hcnt_values` 窗口相位 | chunk 跨内部边界的相位持久化 |
| M3.4 | `recurrent_gdn_leapquant` | decode 内核: steady / payload-init / NULL 行 |

外加: 性能微基准 (torch 三步 flush vs 融合内核; decode 内核) 与 64 步 mini e2e
(对照 A3 e2e 腿锚点: gsm8k 49.5%, mcq 73/200 与 AscendC 基线零翻转, 墙钟 +1.5%)。

**注意**: 需要 A3 NPU; 首次内核调用触发 CANN/BiShengIR JIT 编译 (数分钟), 之后走缓存。

In [1]:
import sys, time, math
import torch

sys.path.insert(0, "/workspace/y00924007/vllm-ascend")  # 开发树

import torch_npu  # noqa: F401
DEV = torch.device("npu")
import warnings, logging
warnings.filterwarnings("ignore")
logging.getLogger("vllm").setLevel(logging.ERROR)

torch.manual_seed(20261110)

try:
    import triton
    tri_ver = triton.__version__
except Exception:
    tri_ver = "?"
print(f"torch = {torch.__version__}   torch_npu = {torch_npu.__version__}   triton = {tri_ver}")

from vllm_ascend.ops.triton.leapquant import leapquant_flush_triton, recurrent_gdn_leapquant
from vllm_ascend.quantization.methods.mamba_state.leapquant.payload import LeapQuantPayloadPool
from vllm_ascend.quantization.methods.mamba_state.leapquant.state import leap_window_flush
print("imports ok — 首次内核调用将触发 JIT 编译 (数分钟), 之后有编译缓存")

/usr/local/python3.11.10/lib/python3.11/site-packages/torch_npu/_init/core/optional_features.py:20: UserWarning: On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default. Do not set it to 1 to avoid unexpected errors.
  warnings.warn(


torch = 2.10.0+cpu   torch_npu = 2.10.0.post7   triton = 3.6.0
INFO 11-10 18:56:53 [__init__.py:52] Available plugins for group vllm.platform_plugins:


INFO 11-10 18:56:53 [__init__.py:54] - ascend -> vllm_ascend:register


INFO 11-10 18:56:53 [__init__.py:57] All plugins in this group will be loaded. Set `VLLM_PLUGINS` to control which plugins to load.


INFO 11-10 18:56:53 [__init__.py:272] Platform plugin ascend is activated


INFO 11-10 18:56:53 [platform.py:85] Breakable CUDAGraph on Ascend is opt-in; using VLLM_USE_BREAKABLE_CUDAGRAPH=0.


imports ok — 首次内核调用将触发 JIT 编译 (数分钟), 之后有编译缓存


## 1. 融合 flush 内核 vs torch golden (M3.1)

同一段数学两种执行: PR1 torch = gather → ~15 个算子 → scatter; triton = **两次内核启动**
(scales 先行, reconstruct 复用存储的 s_k/s_v 保证两遍 bit 一致)。round-trip 模式 (`payload=None`)
直接原地重写 fp32 池行 — PR2 的精度基准与永久 fallback。

In [2]:
SLOTS, H, V, K, R, WINDOW = 16, 8, 128, 128, 4, 16

pool = torch.randn(SLOTS, H, V, K, device=DEV) * 0.3
due  = torch.tensor([3, 7, 11], device=DEV, dtype=torch.int64)

golden = leap_window_flush(pool[due].clone(), compensator_tokens=R)   # PR1 torch 参考

kern = pool.clone()
t0 = time.perf_counter()
leapquant_flush_triton(kern, due, compensator_tokens=R)               # 首调含 JIT 编译
torch.npu.synchronize()
print(f"首调 (含 JIT 编译): {time.perf_counter() - t0:.1f}s")

rel = ((kern[due] - golden).norm() / golden.norm()).item()
mx  = (kern[due] - golden).abs().max().item()
print(f"due 行  kernel vs torch golden:  相对误差 {rel:.3e}   max|Δ| {mx:.3e}")
other = [i for i in range(SLOTS) if i not in due.tolist()]
print("非 due 槽逐位未动:", all(torch.equal(kern[i], pool[i]) for i in other))

[W1110 18:56:59.400658964 NPUCachingAllocator.cpp:204] Warning: The current CANN and HDK(driver) versions require processing for 32 padding size, with memory allocation. (function operator())


首调 (含 JIT 编译): 0.2s
due 行  kernel vs torch golden:  相对误差 3.621e-05   max|Δ| 6.816e-03
非 due 槽逐位未动: True


## 2. payload 池 (M3.2) — ~1.19 B/elem 的状态 checkpoint

payload 模式把 int8 checkpoint (codes + 双尺度 + fp16 补偿平面 + hcnt) 持久化进 `LeapQuantPayloadPool`
(uint8 缓冲区的分域视图), 相对 fp32 状态 4 B/elem 的理论压缩比:

- `rewrite_state=False`: 纯 payload 模式, fp32 行不动 (存储节省模式);
- `rewrite_state=True`: 双写 — fp32 行同时被 round-trip 重写 (迁移安全门, M3.3/M3.4 e2e 的实跑语义)。

In [3]:
pool_pl = LeapQuantPayloadPool.allocate(SLOTS, H, V, K, R, device=DEV)
lay = pool_pl.layout
bpe = lay.bytes_per_element
print("字段偏移 (head 块内):", lay.field_offsets())
print(f"fp32 状态 4.000 B/elem → payload {bpe:.3f} B/elem   压缩 {4.0/bpe:.2f}×")
print(f"每 (slot,head) 块 {lay.head_block_bytes} B = codes {lay.codes_bytes} + s_k {lay.s_k_bytes} "
      f"+ s_v {lay.s_v_bytes} + u {lay.u_bytes} + q {lay.q_bytes} + hcnt 4")

src = torch.randn(SLOTS, H, V, K, device=DEV) * 0.3
snap = src.clone()

# 纯 payload 模式
leapquant_flush_triton(src, due, compensator_tokens=R, payload=pool_pl, rewrite_state=False)
err_ck = ((pool_pl.reconstruct_rows(due) - snap[due]).norm() / snap[due].norm()).item()
print(f"\nrewrite_state=False:  fp32 行逐位未动: {torch.equal(src[due], snap[due])}")
print(f"                       checkpoint 误差 (reconstruct_rows vs 原状态): {err_ck:.3%}")

# 双写模式
src2 = snap.clone()
leapquant_flush_triton(src2, due, compensator_tokens=R, payload=pool_pl, rewrite_state=True)
gold2 = leap_window_flush(snap[due], compensator_tokens=R)
print(f"rewrite_state=True:    fp32 行 vs torch round-trip max|Δ| = "
      f"{(src2[due] - gold2).abs().max().item():.2e}  (与 §1 kernel↔torch 舍入序差异同量级, 非 bit 级)")
print(f"                       payload 读回 vs 双写 fp32 行: max|Δ| = "
      f"{(pool_pl.reconstruct_rows(due) - src2[due]).abs().max().item():.2e}")

字段偏移 (head 块内): {'codes': 0, 's_k': 16384, 's_v': 16896, 'u': 17408, 'q': 18432, 'hcnt': 19456}
fp32 状态 4.000 B/elem → payload 1.188 B/elem   压缩 3.37×
每 (slot,head) 块 19460 B = codes 16384 + s_k 512 + s_v 512 + u 1024 + q 1024 + hcnt 4

rewrite_state=False:  fp32 行逐位未动: True
                       checkpoint 误差 (reconstruct_rows vs 原状态): 0.617%
rewrite_state=True:    fp32 行 vs torch round-trip max|Δ| = 7.23e-03  (与 §1 kernel↔torch 舍入序差异同量级, 非 bit 级)
                       payload 读回 vs 双写 fp32 行: max|Δ| = 5.96e-08


## 3. 窗口相位 hcnt (M3.3)

chunked prefill 的 chunk 可能跨内部窗口边界而不落在边界上 — flush 发生在 chunk 末, `hcnt` 记录
`seq_len % window` (0 = 恰在边界), 读出侧由此知道 checkpoint 折叠到窗口的哪一相位。

In [4]:
# 三个 due 槽分别处于: 恰在边界(相位0) / chunk 末在窗口内第 5 步 / 第 11 步
phases = torch.tensor([0, 5, 11], dtype=torch.int32, device=DEV)
src3 = snap.clone()
leapquant_flush_triton(src3, due, compensator_tokens=R, payload=pool_pl,
                       rewrite_state=True, hcnt_values=phases)
print("pool.hcnt[due] (每槽 × 前 4 头):")
print(pool_pl.hcnt[due][:, :4])
print("语义: readout 侧由 hcnt 知道该 checkpoint 已折叠 seq_len % window 步; 纯边界 flush 存 0")

pool.hcnt[due] (每槽 × 前 4 头):


tensor([[ 0,  0,  0,  0],
        [ 5,  5,  5,  5],
        [11, 11, 11, 11]], device='npu:0', dtype=torch.int32)
语义: readout 侧由 hcnt 知道该 checkpoint 已折叠 seq_len % window 步; 纯边界 flush 存 0


## 4. 递推 decode 内核 (M3.4) — steady / payload-init / NULL 行

`recurrent_gdn_leapquant`: 一步 gated delta rule 的融合内核 (无 tl.dot 的 5 行递推, fp32 累加)。
三种初始化路径:

- **steady**: 从工作区行 (fp32 状态) 继续递推 — 逐位对齐 torch 参考;
- **payload-init**: 冷工作区, 内核从 payload 池内联重建 Ŝ₀ (eq.8-9) 再递推 — 误差 = checkpoint 级;
- **NULL 行** (`w_rows = -1`): 以零状态起步正常递推一步, 但跳过 W 的读和写 (对应生产里未分配的槽)。

In [5]:
SCALE = K ** -0.5
Bt = 4

def step_inputs(B, H, K=K, V=V):
    q = torch.nn.functional.normalize(torch.randn(B, 1, H, K, device=DEV), dim=-1).contiguous()
    k = torch.nn.functional.normalize(torch.randn(B, 1, H, K, device=DEV), dim=-1).contiguous()
    v = torch.randn(B, 1, H, V, device=DEV).contiguous()
    g = -torch.rand(B, 1, H, device=DEV).contiguous() * 0.1
    beta = torch.rand(B, 1, H, device=DEV).contiguous().sigmoid()
    return q, k, v, g, beta

def ref_step(s0, q, k, v, g, beta):
    """torch fp32 参考: 与内核相同的 5 行递推。"""
    h = s0 * g[:, 0].exp()[..., None, None]
    v_adj = (v[:, 0] - torch.einsum("bhvk,bhk->bhv", h, k[:, 0])) * beta[:, 0][..., None]
    h = h + torch.einsum("bhv,bhk->bhvk", v_adj, k[:, 0])
    o = torch.einsum("bhvk,bhk->bhv", h, q[:, 0]) * SCALE
    return h, o

q, k, v, g, beta = step_inputs(Bt, H)
rows = torch.arange(Bt, device=DEV, dtype=torch.int64)

# ---- steady 路径 ------------------------------------------------------
s0 = torch.randn(Bt, H, V, K, device=DEV) * 0.3
W  = s0.clone()
out = recurrent_gdn_leapquant(q, k, v, g, beta, scale=SCALE, w_in=W, w_out=W, w_rows=rows)
h_ref, o_ref = ref_step(s0, q, k, v, g, beta)
print(f"steady:        out 相对误差 {((out[:, 0] - o_ref).norm() / o_ref.norm()).item():.2e}"
      f"   状态相对误差 {((W - h_ref).norm() / h_ref.norm()).item():.2e}")

# ---- payload-init 路径 ------------------------------------------------
seed_pool = torch.zeros(SLOTS, H, V, K, device=DEV)
seed_pool[:Bt] = s0
slots = rows.clone()
leapquant_flush_triton(seed_pool, slots, compensator_tokens=R, payload=pool_pl, rewrite_state=False)
W2 = torch.zeros_like(s0)                       # 冷工作区: 行内容不被读, 只作写出目标
out2 = recurrent_gdn_leapquant(q, k, v, g, beta, scale=SCALE,     # 不传 w_in → USE_PAYLOAD_INIT
                               w_out=W2, w_rows=rows, payload=pool_pl, slot_ids=slots)
S0_hat = pool_pl.reconstruct_rows(slots)        # torch 侧读回 checkpoint (eq.8-9)
h_hat, o_hat = ref_step(S0_hat, q, k, v, g, beta)
print(f"payload-init:  内核内联重建递推 vs torch(decode Ŝ₀) {((out2[:, 0] - o_hat).norm() / o_hat.norm()).item():.2e}")
print(f"               W2 (递推后) vs torch(decode Ŝ₀):     max|Δ| = {(W2 - h_hat).abs().max().item():.2e}")
print(f"               out vs 真 S₀ 递推 {((out2[:, 0] - o_ref).norm() / o_ref.norm()).item():.3%}"
      f"   (≈ checkpoint 量化误差级别)")

# ---- NULL 行 ----------------------------------------------------------
rows3 = rows.clone(); rows3[0] = -1
W3 = torch.zeros_like(s0); W3[1:] = s0[1:]
out3 = recurrent_gdn_leapquant(q, k, v, g, beta, scale=SCALE, w_in=W3, w_out=W3, w_rows=rows3)
_, o_zero = ref_step(torch.zeros_like(s0), q, k, v, g, beta)
print(f"NULL 行:       行0 == 零状态起步参考: {((out3[0, 0] - o_zero[0]).norm() / o_zero[0].norm()).item():.2e}"
      f"   行0 未被写回: {bool((W3[0] == 0).all())}"
      f"   其余行正常: {((out3[1:, 0] - o_ref[1:]).norm() / o_ref[1:].norm()).item():.2e}")

steady:        out 相对误差 1.33e-07   状态相对误差 1.75e-08
payload-init:  内核内联重建递推 vs torch(decode Ŝ₀) 1.43e-07
               W2 (递推后) vs torch(decode Ŝ₀):     max|Δ| = 2.38e-07
               out vs 真 S₀ 递推 0.608%   (≈ checkpoint 量化误差级别)
NULL 行:       行0 == 零状态起步参考: 1.45e-07   行0 未被写回: True   其余行正常: 1.34e-07


## 5. 性能微基准

In [6]:
def bench(fn, n=20, warmup=3):
    for _ in range(warmup):
        fn()
    torch.npu.synchronize()
    t0 = time.perf_counter()
    for _ in range(n):
        fn()
    torch.npu.synchronize()
    return (time.perf_counter() - t0) / n

D = 8
pool_t = torch.randn(SLOTS, H, V, K, device=DEV) * 0.3
pool_k = pool_t.clone()
due8 = torch.arange(D, device=DEV, dtype=torch.int64)

def torch_flush_path():                          # PR1 三步: gather → flush → scatter
    pool_t[due8] = leap_window_flush(pool_t[due8], compensator_tokens=R)

def triton_flush_path():                         # PR2: 两次内核启动
    leapquant_flush_triton(pool_k, due8, compensator_tokens=R)

ms_t = bench(torch_flush_path, n=20) * 1e3
ms_k = bench(triton_flush_path, n=50) * 1e3
print(f"flush {D} due × [{H}H {V}×{K}]:  torch 三步 {ms_t:.2f} ms   triton 融合 {ms_k:.2f} ms   → {ms_t / ms_k:.1f}×")

Bd = 8
s0d = torch.randn(Bd, H, V, K, device=DEV) * 0.3
Wd = s0d.clone()
qd, kd, vd, gd, betad = step_inputs(Bd, H)
rowsd = torch.arange(Bd, device=DEV, dtype=torch.int64)
us_dec = bench(lambda: recurrent_gdn_leapquant(qd, kd, vd, gd, betad, scale=SCALE,
                                               w_in=Wd, w_out=Wd, w_rows=rowsd), n=50) * 1e6
us_ref = bench(lambda: ref_step(s0d, qd, kd, vd, gd, betad), n=50) * 1e6
print(f"decode 一步 B={Bd}:           triton 内核 {us_dec:.0f} µs   torch 参考算子串 {us_ref:.0f} µs")

flush 8 due × [8H 128×128]:  torch 三步 6.24 ms   triton 融合 0.34 ms   → 18.3×
decode 一步 B=8:           triton 内核 112 µs   torch 参考算子串 314 µs


## 6. 64 步 mini e2e — M3.4 语义 (decode 内核 + 边界 payload 双写 flush)

每个 decode 步走 triton 递推内核 (工作区行原地更新), 每 16 步对工作区做 payload 双写 flush
(`rewrite_state=True` → 量化误差进入活跃状态 = 论文语义, 与 M3.3/M3.4 e2e 腿一致)。
对照集成 UT 容差: 输出 < 2%, 状态 < 5%。

In [7]:
T, B2 = 64, 2

def seq_inputs(B, H, T):
    q = torch.nn.functional.normalize(torch.randn(B, H, T, K, device=DEV), dim=-1)
    k = torch.nn.functional.normalize(torch.randn(B, H, T, K, device=DEV), dim=-1)
    v = torch.randn(B, H, T, V, device=DEV)
    g = -torch.rand(B, H, T, device=DEV) * 0.1
    beta = torch.rand(B, H, T, device=DEV).sigmoid()
    return q, k, v, g, beta

qs, ks, vs, gs, betas = seq_inputs(B2, H, T)

def gdn_step(h, q, k, v, g, beta):
    h = h * g.exp()[..., None, None]
    v_adj = (v - torch.einsum("bhvk,bhk->bhv", h, k)) * beta[..., None]
    return h + torch.einsum("bhv,bhk->bhvk", v_adj, k)

# fp32 参考
h_ref = torch.zeros(B2, H, V, K, device=DEV)
outs_ref = []
for t in range(T):
    h_ref = gdn_step(h_ref, qs[:, :, t], ks[:, :, t], vs[:, :, t], gs[:, :, t], betas[:, :, t])
    outs_ref.append(torch.einsum("bhvk,bhk->bhv", h_ref, qs[:, :, t]) * SCALE)
outs_ref = torch.stack(outs_ref, dim=2)

# LeapQuant triton 轨迹
pool_e2e = LeapQuantPayloadPool.allocate(SLOTS, H, V, K, R, device=DEV)
W_t = torch.zeros(B2, H, V, K, device=DEV)
rows_b = torch.arange(B2, device=DEV, dtype=torch.int64)
outs_lq = []
for t in range(T):
    o = recurrent_gdn_leapquant(qs[:, :, t][:, None].contiguous(), ks[:, :, t][:, None].contiguous(),
                                vs[:, :, t][:, None].contiguous(), gs[:, :, t][:, None].contiguous(),
                                betas[:, :, t][:, None].contiguous(),
                                scale=SCALE, w_in=W_t, w_out=W_t, w_rows=rows_b)
    outs_lq.append(o[:, 0])
    if (t + 1) % WINDOW == 0:
        leapquant_flush_triton(W_t, rows_b, compensator_tokens=R, payload=pool_e2e,
                               rewrite_state=True, hcnt_value=0)
outs_lq = torch.stack(outs_lq, dim=2)

out_drift = ((outs_lq - outs_ref).norm() / outs_ref.norm()).item()
st_drift  = ((W_t - h_ref).norm() / h_ref.norm()).item()
print(f"64 token / window=16 / r=4:  输出漂移 {out_drift:.2%}   状态漂移 {st_drift:.2%}")
print("对照: 集成 UT 容差 out<2% / state<5%; M3.4 e2e 腿 gsm8k 49.5%, mcq 73/200 零翻转, 墙钟 +1.5%")

64 token / window=16 / r=4:  输出漂移 0.31%   状态漂移 0.56%
对照: 集成 UT 容差 out<2% / state<5%; M3.4 e2e 腿 gsm8k 49.5%, mcq 73/200 零翻转, 墙钟 +1.5%


## 小结

- **M3.1** flush 融合内核与 torch golden 对齐 (相同数学, fp32 累加序差异在 1e-7 量级);
- **M3.2** payload 池把状态 checkpoint 压到 ~1.19 B/elem (fp32 的 ~1/3.4), 双写语义与 torch round-trip 一致;
- **M3.3** hcnt 持久化 chunk 末窗口相位;
- **M3.4** 递推 decode 内核 steady 路径 ~1e-6 级对齐 torch, payload-init 误差收敛到 checkpoint 级, NULL 行安全跳过;
- mini e2e 漂移与实卡集成 UT 同量级。

**展望**: M3.4a 工作区生命周期 (W 影子池 + 丢弃 fp32 页分量 → 池容量 ~3.2×), M3.4b eq.10 环形缓冲纯读出
(状态带宽 ~3.37×↓)。